In [1]:
import copy
import torch 
import pandas as pd
import numpy as np
import pickle as pkl

from pathlib import Path
from tqdm import tqdm 

from neuralhydrology.evaluation import get_tester
from neuralhydrology.utils.config import Config

In [12]:
with open("/home/wuhlmann/BA/data/processed_data/SA/full_q_512_3011_185525_raw_nse_deltas.p", "rb") as f: 
    raw_nse = pkl.load(f)

with open("/home/wuhlmann/BA/data/processed_data/SA/full_q_512_3011_185525_weighted_means_deltas.p", "rb") as f: 
    means = pkl.load(f)

In [18]:
attr_counter = {}

for df in means.values(): 
    
	top_attr = df.index[0]

	if not top_attr in attr_counter: 
		attr_counter[top_attr] = 0
	attr_counter[top_attr] += 1

In [19]:
attr_counter

{'elev_mean': 36, 'area_calc': 41, 'elev_ran': 11}

In [2]:
# Load the config.
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525")
cfg = Config(run_dir_path / "config.yml")


In [3]:
#attribute one. 
tester = get_tester(cfg=cfg, run_dir=run_dir_path, period="test", init_model=True)


In [ ]:
raw_results = tester.evaluate(save_results=False, metrics=["NSE"])

In [ ]:
basin_ids_list = list(tester.cached_datasets.keys())
baseline_nse_values = np.array([raw_results[id]["1D"]["NSE"] for id in basin_ids_list])
attr_ids = [0]
noise_amounts = [-0.5,-0.1, 0.1, 0.5]

num_basins = len(basin_ids_list)
num_attr = len(attr_ids)
noise_levels = len(noise_amounts)

# 3D array with dim basin x attribute x noise_level, to hold raw numeric values
raw_array = np.zeros([num_basins, num_attr, noise_levels])

for attr_id in attr_ids:
	
	for noise_id in range(noise_levels): 

		# restore original attributes values in tester
		tester_copy = copy.deepcopy(tester)

		# add noise to the attribute in every catchment
		for id in basin_ids_list:	
			tester_copy.cached_datasets[id]._attributes[id][attr_id] += noise_amounts[noise_id]

		# run evaluation for the currrent noise level
		tester_result_dict = tester_copy.evaluate(save_results=False, metrics=["NSE"])
		nse_values = np.array([tester_result_dict[id]["1D"]["NSE"] for id in basin_ids_list])

		# write NSE for all basins, for the current attribute and noise level	
		raw_array[:, attr_id, noise_id] = abs(nse_values - baseline_nse_values)

In [ ]:
results_dict = {}

for i in range(len(basin_ids_list)): 
	
	results_dict[basin_ids_list[i]] = pd.DataFrame(data=raw_array[i,:,:], index=[cfg.static_attributes[0]], columns=noise_amounts)
	

In [ ]:
test_df = results_dict["116"]
test_df

In [ ]:
attr_ranks = dict(zip(test_df.index, [0]*len(test_df.index)))

attr_weights = [0.75, 1, 1, 0.75]

for col, weight in zip(test_df.columns, attr_weights):
	
	order = list(test_df.sort_values(by=col, ascending=False)[col].index)

	for attr in test_df.index: 

		attr_ranks[attr] += (order.index(attr)+1) * weight



In [ ]:
for id in results_dict.keys():

	print(f"{id} {'-'*10}")

	basin_df = results_dict[id]

	attr_ranks = pd.DataFrame(data=[0]*len(basin_df.index), index=basin_df.index, columns=["rank"], dtype="float")

	attr_weights = [0.75, 1, 1, 0.75]

	for col, weight in zip(basin_df.columns, attr_weights):
		
		order = list(basin_df.sort_values(by=col, ascending=False)[col].index)

		for attr in basin_df.index: 

			attr_ranks.loc[attr, "rank"] += (order.index(attr)+1) * weight

	mean_attr_ranks = attr_ranks.apply(lambda x: x/4)

	print(mean_attr_ranks.sort_values(by="rank", ascending=True))
	